## ATracker Tutorial

ATracker works with folders of videos, or better said, projects. It was designed to handle large folders of videos to be tracked with many different parameters that may furthermore differ between videos, such as having refuges or not, the number of objects to track etc.

This Jupyter notebook provides a **step-by-step tutorial** on using ATracker, covering everything from loading videos to tracking and processing data. We will work with **six example video files** designed specifically for this tutorial. These files will guide you through:
- Drawing the **region of interest (ROI)**
- Creating and adjusting **masks** for specific areas (e.g., refuge zones)
- Setting up **zones** for tracking structures in the arena
- Defining **walls** for circular arenas
- Configuring different **thresholding methods** for Background subtraction tracking, Color-based tracking, and  Barcode tracking (to implement).
- Running **drymode** for testing tracking parameters
- Running tracking
- And manual tracking mode

### 1. Setting Up

We start by initiating the video folder. You can type or drag a folder path into Python on any system. Just pass it to ATracker() like this:

In [ ]:

AT = atracker.ATracker("C:/Users/me/Desktop/tutorial")
AT = atracker.ATracker(r"C:\Users\me\Desktop\tutorial")
AT = atracker.ATracker("/Users/me/Desktop/tutorial")
AT = atracker.ATracker("dragged_path_here")

For this tutorial, to ensure that we work with a clean dataset without modifying the original tutorial files:

1. **Copy the example video files** (only the `.mp4` files) from the `0originals/` folder.
2. **Paste them into a new folder** called `"test"` on your **Desktop**.

Then run:

In [ ]:
import os
import atracker

# Get the user's home directory
home_dir = os.path.expanduser("~")

# Define the video folder path (inside "test" on the Desktop)
video_folder = os.path.join(home_dir, "Desktop", "tutorial")

# Load ATracker with the video folder
AT = atracker.ATracker(video_folder)

ATracker will have initiated the folder for tracking, including creating a number of files, about which it prints some statements in the output window. The folders it creates are as follows:

- **0originals**: stores all raw files used for tracking. This includes the raw .h264 and converted .mp4 videos but also for example background files or mask files. mp4 files are automatically moved to this folder, ready for tracking.
- **1todo**: for (temporally) storage of videos that need to be tracked. You can for example decide to track a couple specific videos, the you can simply drag them there and in python tell tracking to look at the todo folder (explained below).
- **2temp**: is a folder only used by atracker. It moves relevant files it needs to tracking to this folder during tracking, and after finishing it moves the files again.
- **3tracked**: is the folder that contains all csv tracking files as well as the output videos from tracking.
- **4processed**: contains all the csv files created from the tracked csv files using the processing function.

Besides these folders, you will see a couple new files:

- **_config.conf**: simple text file that contains all relevant parameters used for tracking. These configuration settings can be manually changed here or using the `set_config` function.
- **_overview.xslx**: a template excel file to fill with relevant data about all the videos. All the videos we want to track should be in this table. We can easily add this list as well as extract all relevant information using the `setup_files` function (explained below)
- **_threshinfo.yml**: for storing all relevant information regarding the objects to track and is mainly used by atracker automatically. However, the user can also use this file to make quick and simple changes to the threshold settings.

### 2. Setting up the video files

The next thing to do is to load the video files. MP4 videos are already automatically moved to the "0 originals" folder, but they are not yet in the overview file. To add them while also extracting relevant information, such as on video width, height, fps, and duration we use the `setup_files` function.

This function takes a number of parameters to help extract the right variable names from the filename and potentially convert the videos as well:
-  `fname_extract`: if the variables should be extracted from the filename (True/False)
-  `fname_vars`: list of variables to be extracted (a tuple of strings)
-  `fname_sep`: the charater that separates the components of the filename (a string)
-  `autoconvert`: if videos should be automatically converted to .mp4 (True/False)
-  `skip`: if videos already in the overview file should be skipped from loading (True/False)

For example, if I have files with a specific filename structure such as `tempexp_250523_setup1_143024_S01.h264`, I can automatically extract the variables and add them to the overview file as follows:

In [ ]:
AT.setup_files(fname_vars = ("exp","name","date","session","time"), fname_sep = "_")

If videos are recorded with Raspberry Pi, such as using the **pirecorder** package, then these will be .h264 format. These can thus be automatically converted to .mp4 using the `autoconvert` parameter, which is True by default.

To only convert videos without extracting filenames you can do the following:

In [ ]:
AT.setup_files(fname_extract=False, autoconvert = True)

You can now check the overview excel file to indeed see the five videos are added as well as relevant video information and the variables are added to the respective columns. You can also access it directly in Python via the following command:

In cases where we update the excel file directly while being in an active Python session, we need to make sure it is updating the loaded overview. To do this use the `reload` function:


In [ ]:
# After having made changes to the excel file directly, load the updated overview:
AT.reload()

In cases where you make updates to the overview in Python and want them to be stored for later:

In [ ]:
AT.save()

To show the overview information for a single video, the `showinfo` function can be used:

In [ ]:
AT.showinfo("animtest_regions_071024_S02_1205")

### 3. Listing files and getting file indices

To get a list of videos with their indices, the `get_files` function can be used. A couple special parameters can be used to select only specific videos. This helps enormously when needing to only update the roi on different days or the mask based on the setup, or to set different threshold parameters for files in the morning versus afternoon. For this we can use the `inds`, `query`, and `cats` parametes as explained below.

To simply show all files with full path and their indices you can use the get_files function:

In [ ]:
AT.get_files()

We can also get the full paths of the files as if they were in a different folder, such as in the temp directory, using the cdir parameter. For example:

In [ ]:
AT.get_files(cdir="temp")

In a similar way we can change the filetype in the generated list of names, for example to change .mp4 to .csv:

In [ ]:
AT.get_files(filetype=".csv")

These above functions are particularly handy in the context to check if files exist. For this you can use the `exists_only` parameter and set it to `True`. For example, to check what files of the overview exist in the tracked folder:

In [ ]:
AT.get_files(cdir="tracked", filetype=".csv", existonly=True)

We can also simply show a subset of files based on their indices:

In [ ]:
AT.get_files(inds=[2,3,4])

If we want the opposite, i.e. get the indices of files, we can use the get_inds function:

In [ ]:
AT.get_inds("animtest_regions_071024_S02_1205")

Another way to select files is to use the `query` parameter and use column names and criteria. For example, we can use the skip column to only select videos that should not be skipped"

In [ ]:
AT.get_files(query="skip==0")

or use the "time" column to only select videos taken before a certain time (make sure the column is converted first):

In [ ]:
AT.overview['time'] = AT.overview['time'].astype(int)
AT.get_files(query="time <= 1300")

Finally we can subset based on categories. This is very handy as it will automatically select the first video of all videos that are unique in terms of that category. For example, if there are 6 videos of two trials, it will show the video information of two videos, the first of each trial. This is helpful if video settings remain the same within the category and thus for example the roi only needs to be set for each session:

In [ ]:
AT.get_files(cats=["session"])

### 4. ATracker configuration

#### 4.1 Set the configuration file

We can set a lot of parameters for all the different elements to do with tracking. These are all stored dynamically in the python session, but also in the `_config.conf` file, so they can be easily manually updated without having to open python.

To configure the tracking and other settings of atracker in Python the `set_config` function can be used. The documentation explains nicely everything that can be set and in what way:

In [ ]:
print(AT.set_config.__doc__)

We can also just show the current configuration settings:

In [ ]:
print(AT.config)

When setting up your project, it may be handy to set a number of settings, but we can do this also step by step as we go through the different parts of the tracking process. Here is an example of how to set the configuration:

In [ ]:
AT.set_config(fps=25, startframe=50, orient_get=False, show_tracking=False, userwait=False, 
              frame_disstep=100, overwrite=False)

#### 4.2 Set number of objects

By default, the number of objects that will be tracked is 1. If we want to change this, we can set the `cusobjects` parameter in the `track` function directly when tracking. In case videos differ in the number of objects that need to be tracked (such as some videos have one object, others have 2 etc) it is best to set that directly in the *objects* column of the overview file. 

We can set the objects directly in the excel file and then reload it (see above) or by using the dedicated `set_objects` function. For example, for our tutorial:

In [ ]:
AT.set_objects(inds=[0,4], objects=[4,1])

#### 4.3 Extract background files

ATracker relies on background substraction for many functionalities. Although some types of tracking do not need it, it is helpful in most cases to have background images for all videos where the animal is not visible. We can do this with the `get_bgfiles` function.

In [ ]:
AT.get_bgfiles(overwrite=False)

In most cases the default parameters are fine, namely it will randomly select 25 frames in the whole video to remove any moving objects and thus only keep the real background. But in some cases we may want to use less or more images to get a more reliable background image. This should be set in the configuration under the `bg_frames` parameter. For example, to use 10 images only, use:

In [ ]:
AT.set_config(bg_frames=10)
AT.get_bgfiles()

We can also set the start and stop frames to use for background substraction. For example, sometimes there might be a lot of movement at the start and end of the video that we want to ignore. We can set this with the `starts` and `stops` parameters. For example, to get bgfiles for all videos but within frame 500 to 5000:

In [ ]:
AT.get_bgfiles(starts=[500], stops=[5000])

Finally, we can use the `overwrite` parameter to overwrite background images if needed. 

The newly created background files are automatically stored in the `0originals` folder as well as listed in bg`bgimg` column of the overview file.

#### 4.4 Setting up videos that have multiple separate tracking regions

With ATracker it is possible to track the same video multiple times to track different regions within the video. But since all the tracking file information will be the same (except the roi), some extra information needs to be provided to ATracker that it knows it should check for regions. We do this using the `set_config` function by setting the `regions` parameter to True:

In [ ]:
AT.set_config(regions=True)

Now a new column with `region` is added to the overview file. If regions is set to `False`, the column, if it exists, will be deleted again such that ATracker will not try to look for region information later.

The next step is to add the region information to the overview file. For this we use the `set_regions` function, which takes the parameters `inds` and `nr`, which is a list of indices of the file(s) in the overview to create the regions for and the number of regions to create. For example, for the tutorial, video 4 (ind=3) has four regions:

In [ ]:
AT.set_regions(inds=[3], nr=4)

and let's look at the change in the overview file

In [ ]:
AT.overview.loc[:,["video","region"]]

Now ATracker is set up to track each video multiple times corresponding to the different regions, and tracked videos with different regions  automatically get the region number appended to the file name e.g. `vidx243_R1.mp4`, `vidx243_R2.mp4`. 

Note that if the number of regions differs between videos, you can either call the `set_regions` function in a loop or add the information manually in the overview file and reload it (`AT.reload()`).


### 5. Setting key parameters interactively using the visual editor


The `set_interactive` function launches ATracker’s **visual editor**, a flexible graphical tool to annotate key tracking parameters. It can be used for setting framelimits, ROI, masks, zones, thresholding parameters, and more — all using intuitive drawing and clicking interactions.

Once opened, the interface displays a live view of the video (or image), combined with a series of interactive panels depending on the selected mode. These may include:

- **Drawing panel**: Draw on the video using shapes like points, lines, rectangles, ellipses, polygons, and circles. Simply click or drag with the mouse to define shapes.
- **Video control panel** (when using videos): Scroll through frames, play/pause, or jump to start/end frames using buttons or keyboard shortcuts.
- **Info panel**: Shows current frame number, mouse coordinates, shape details, and optionally contour info for thresholding modes.
- **Image processing panel**: Tools to adjust drawing hue, convert the image to black-and-white, and more.
- **Thresholding panel**: For tuning parameters in background subtraction or color-based tracking modes, including contour filtering and preview images.
- **Timepoint control panel**: (in timepoints mode) to control the number of tracked IDs, selected ID, frame range visibility, and point editing options.

The mouse can be used not only to draw, but also to **edit points** interactively (by enabling nearest-point dragging), and to inspect frame-based or shape-based data.

Each mode customizes the interface to match its task — hiding irrelevant panels and showing only what’s needed. Below we describe each parameter you can set interactively, and how it works within the visual editor.

To just try out the visual editor just run the following command and it will show the default mode:

In [ ]:
AT.set_interactive()

Note: The visual editor is a core component of ATracker, integrated directly through `AT.set_interactive()`. However, it can also be launched independently (without the ATracker pipeline) via:

In [ ]:
atracker.visual_editor.annotation_gui(media_file="~/Desktop/sample_video_col.mp4", 
                                      datafile="/Users/Jolle/Desktop/annotated_timepoints.csv")

This standalone mode is useful for quickly annotating or exploring videos and images without needing the full ATracker configuration.

#### 5.1 Video Controls

In any video-based mode, video navigation tools are shown. The following keyboard shortcuts are available:

| Key | Action |
|-----|--------|
| `q` | Jump to the first frame of the video |
| `w` | Move backward by 1 second |
| `e` | Move backward by 1 frame |
| `r` | Move forward by 1 frame |
| `t` | Move forward by  1 second |
| `y` | Jump to the last frame of the video |
| `space` | Play/pause the video (toggle) |
| `s` | Save the current results and exit |
| `Esc` | Exit file without saving (double quick escape to exit all) |
| `a` | Add current shape to mask |
| `z` | Store current drawing as a new zone (only in "zones" mode) |
| `d` | Delete the currently drawn shape, mask, zones or point |
| `i` | Switch to next object ID (only in timepoints mode) |
| `h` | Toggle helperlines |
| `c` | Toggle crosshair |

#### 5.2 Interactive modes

Only a single mode should be set as `True`, such that the interface options are automatically adjusted to show only what is required.

Each mode opens the graphical interface with relevant tools visible: video panel, shape tools, sliders, info panels, and background or mask displays depending on the mode.

| **Parameter**  | **Purpose** |
|----------------|-------------|
| `conv`         | Draw one or more known-length lines to compute pixel-to-mm conversion. Use `conv_mm=(length,)` or provide multiple values. |
| `framelimits`  | Use the frame slider and buttons to set start and stop frames. |
| `roi`          | Draw a rectangular region of interest by clicking and dragging. |
| `mask`         | Draw exclusion areas using any shape (rectangle, polygon, ellipse, etc.). Add with `a`, show with `m`, invert with `i`. |
| `maskzone`     | Same as `mask`, but saved separately to distinguish different types (e.g. refuge vs structure). |
| `walls`        | Mark boundaries of tanks/arenas using shape tools. Stored as a separate mask. |
| `zones`        | Define multiple distinct zones. After drawing, press `a` to add to mask, then `o` to store the zone. Saved as a multicolor zone image. |
| `getpts`       | Mark one or more points of interest. Saved to the overview table. Optionally name with `ptcolnames`. |
| `threshtypes`   | Calibrate threshold parameters. Use `"bw"` for grayscale or color names like `"red"`, `"green"`, etc. |

A great functionality is that we can set specific parameters for specific videos using the `inds`, `query`, and `cats` parameters. For example, we may need to create a specific mask for each setup due to small differences in camera position. Or we may run two sessions per day and each session has a different position of plants. In those cases we can nicely and efficiently set parameters for the minimum set of videos.

Here are two quick examples showing how to launch the interface in different modes:

In [ ]:
# Draw a mask and save it
AT.set_interactive(mask=True, inds=[0])

# Draw a line to calibrate distance (300mm)
AT.set_interactive(conv=True, conv_mm=(300,))

#### 5.3 Set Pixel-to-mm Conversion

To translate pixel distances into real-world distances, ATracker offers two main ways to compute conversion ratios:

1. **Use known dimensions from the config file**  
   If the region of interest represents a known real-world area (e.g., 300×200 mm), you can provide this via the `real_dims` configuration. The values will be used to automatically compute the pixel-to-mm ratio after drawing the region.

2. **Draw a known-length line and provide real distance**  
   Alternatively, draw one or more lines in the video that correspond to known physical lengths. Provide the real-world values using the `conv_mm` parameter as a tuple or list. Each time you draw a line and press `s`, it links the last-drawn line with the corresponding real-world value.

In [ ]:
# Draw two known-length lines and link to real-world distances in mm
AT.set_interactive(conv=True, conv_mm=(300), inds=[2])

#### 5.4 Framelimits


By default, ATracker uses the full video range as the start and stop frame limits. However, it is often useful to restrict the range used for tracking — for example, if part of the video is irrelevant or empty.

To do this, set the `framelimits` parameter to `True`. This opens the video in the interactive mode and allows you to use the video frame slider to choose the start and end frames. Once the desired range is selected:

- Click **‘Set Start’** to mark the current frame as the start.
- Click **‘Set Stop’** to mark the current frame as the stop.
- Press **‘s’** to save the selected frame range to the overview.

In [ ]:
AT.set_interactive(framelimits=True, inds=[2])

#### 5.5 Region of Interest


To restrict tracking to a specific area of the video, you can set a region of interest (ROI). This is useful when only a part of the frame contains relevant behavior or when tanks or arenas don’t fill the entire view.

To define the ROI, set the `roi` parameter to `True`. The interactive tool will open with the drawing tool set to **rectangle** mode. Click and drag to draw a rectangle over the area you want to keep.

- Press **‘s’** to save the drawn rectangle.
- The coordinates will be stored in the `roi` column of the overview file.

In [ ]:
AT.set_interactive(roi=True, inds=[2])

For example, for the video with the four regions we would drag 4 different roi's like:

In [ ]:
AT.set_interactive(roi=True, inds=[3,4,5,6])

#### 5.6 Mask

To create a mask, you can draw one or more shapes on the video using the shape tools (rectangle, polygon, ellipse, etc.). These masks define areas to exclude from tracking (e.g. plant refuges, shadows).

Once a shape is drawn, press `a` to add it to the current mask layer. You can continue adding more shapes as needed.

Use the **checkboxes in the interface** to visualize the mask or to invert it (i.e. exclude the area *outside* the drawn region).

The final mask will be saved as a black-and-white `.jpg` file in the `originals` directory, where black represents excluded areas.

In [ ]:
# Draw a mask for the area of the refuge, which will be excluded for tracking 
AT.set_interactive(mask=True, inds=[6])

#### 5.7 Walls

The `walls` parameter allows you to manually draw the exact outline of the tank or arena walls. This is especially useful when arenas are **non-rectangular**, **tilted**, or **circular**, or when additional structural elements (like partial barriers) need to be accounted for.

You can use any of the available shape tools (e.g. rectangle, polygon, circle) to draw the wall boundary. If needed, use the interface controls to **add multiple shapes** (via the `a` button) and **preview the mask** using the checkboxes.

Importantly, the area to be tracked should be white, so in case the walls surround a (part of) the tracking area, the mask should be inverted as well.

Once saved, the result is stored as a black-and-white mask image in the `originals` folder. This can later be used to compute things like **distance to wall**, **position within tank**, or to constrain movement tracking within physical boundaries.

In [ ]:
# Draw the walls of a circular arena to later calculate distance to wall
AT.set_interactive(walls=True, inds=[10])

#### 5.8 Maskzone

The `maskzone` parameter works exactly like `mask`, allowing you to draw one or more shapes and store them as a black-and-white image. However, the key difference is that this mask is stored separately — typically used to highlight **alternative areas of interest** (e.g. structural elements, shelters, barriers) while keeping the primary `mask` reserved for excluded regions.

You can draw using any of the shape tools, click **Add** (`a`) after each shape, and press **Save** (`s`) when finished. The resulting mask will be stored as a grayscale image in the `maskzone` column of your `AT.overview`.

In [ ]:
# Draw a second mask for refuge areas or structural elements
AT.set_interactive(maskzone=True, inds=[9])

This is helpful when you want to distinguish between different spatial features without combining them in a single mask file.

#### 5.9 Zones

The `zones` parameter allows you to create multiple distinct spatial zones within the region of interest. This is particularly useful when you want to analyze behavior in relation to **specific areas**, such as shelters, landmarks, or compartments.

Each zone is created by drawing a shape (e.g. rectangle, polygon, ellipse) and adding it to the zone stack using the **Add** (`a`) button. After adding, press the **Store Zone** (`z`) button to commit the current shape as a separate zone. You can then continue drawing more zones using the same method. You can delete all zones with the `d` key.

All zones are saved into a **single multicolor zone mask**, where each zone has a unique color. Note that if zones overlap, the most recently added zone takes precedence in the saved image. When a zones image already exists that image will automatically be loaded.

In [ ]:
# Use the interactive editor to define multiple zones (e.g. different foraging areas)
AT.set_interactive(zones=True, inds=[10])

#### 5.10 Points of Interest


The `getpts` parameter allows you to manually place specific points on the video frame — useful for calculating distances to fixed locations like shelters, feeders, or corners.

Each click adds a point, and once you're finished placing all desired points, press **Save** (`s`). These points will be saved to the `AT.overview` table, with each point stored in a separate column.

Optionally, you can define the names of these columns using the `ptcolnames` parameter. Make sure the number of names matches the number of points you plan to place.

This functionality is ideal for spatial metrics like **distance to shelter** or **time spent near a cue**, e.g.

In [ ]:
AT.set_interactive(getpts=True, ptcolnames=["refuge", "stone1", "stone2"], inds=[10])

#### 5.11 Thresholding parameters

To optimize object detection for tracking, the `threshtypes` parameter opens the visual editor in **thresholding mode**. This allows you to calibrate either background subtraction (`"bw"`) or **color-based tracking** (e.g. `"red"`, `"green"`, etc.).

When launched, the interface includes:

- A **video panel** with frame-by-frame navigation
- A **thresholding control panel** with sliders for:
  - `blur`, `blur2`: smoothing filters
  - `erode`: reduces contour size
  - `threshold`: sets the brightness difference threshold
  - `min_area`, `max_area`: filters object sizes
- A **preview panel** showing the thresholded image
- A **contour info box** listing detected object sizes

Contours that **match** current filter settings are shown in **blue**, while those that fail size or shape filters are shown in **red**. You can explore how the settings behave across frames using the video controls. It is thus key to make sure your objects of interest show in blue, not red.

To hide or show different views:
- **Toggle threshold image** with the checkbox: *Show Thresholded Image*
- **Toggle contour overlay** using: *Show Overlay*
- **Enable grayscale view** via the *Black & White* option

If you're using **color thresholding**, an additional **HSV panel** appears with sliders for:
- Hue, saturation, and value ranges
- A preview of the current hue window

The goal is to identify and fine-tune settings so that **only the objects of interest are detected** consistently throughout the video.

Once you're satisfied, press **Save** (`s`) to store the values. These are saved to a special YAML file (`threshinfo.yml`), and linked to each video via the `thresh_types` column of the `AT.overview` file. You can later also update this file manually using a simple text editor.

Here's how to use it for black-and-white thresholding. Key is that rather than for the othter modes of set_interactive, when setting the treshtypes we only set treshtypes once and in a dedicated "_treshinfo.yml" file, so do NOT use the `query` or `cats` command, and when using multiple `inds` remember that it will overwrite the values, it is just handy that way to make sure your values work across those videos.

In [ ]:
# Set threshold parameters for black and white tracking
AT.set_interactive(threshtypes=["bw"], inds=[1])

And for color thresholding:

In [ ]:
# Set multiple color threshold parameters in series
AT.set_interactive(threshtypes=["red","blue","green","orange","black","brown"], inds=[1])

In case you want to set **alternative parameters** for black-and-white thresholding, simply name your threshold type starting with `"bw_"` followed by a custom label. For instance:

In [ ]:
AT.set_interactive(threshtypes=["bw_lightissue"], inds=[5])

This tells the system to treat it as black-and-white thresholding, while allowing separate tuning from the default `"bw"`.

You can also create separate threshold files that store different threshtypes by using the `threshfile` parameter:

In [ ]:
AT.set_interactive(threshtypes=["bw"], inds=[5], threshfile="threshinfo_smallfish")

To link a video to a specific thresholding profile, you should set the `thresh_types` column in your `AT.overview` table. For example:

In [ ]:
AT.overview.loc[:, ["video", "thresh_types"]]

By default, if the column is empty for a video, it uses `"bw"`. But you can explicitly define which threshold profiles to use per video or per region. These values should be a list of strings. For example:

In [ ]:
AT.overview.thresh_types = ["(red,green)", "bw", "blue", "bw_strict"] + ["bw"] * 6
AT.save()

This ensures each video is tracked using the right thresholding settings, whether based on color or grayscale subtraction.

### 6. Batch Measuring Functionality in ATracker

ATracker includes batch measuring functionality, allowing you to quickly measure lengths (such as distances or objects) across an entire folder of images or videos using its interactive annotation GUI. This workflow is ideal for collecting consistent measurement data for large datasets, including morphometrics, calibration, or experimental results.

To use it:

1. **Prepare your data:**  
   Place all images or videos to be measured in a single folder. Supported formats include JPG, PNG, BMP, TIFF, MP4, AVI, and more.

2. **Launch the batch measurement tool:**  
   You can run the batch measuring script either from the command line or directly within Python. For command-line use, run:

In [ ]:
python batch_measure.py /path/to/folder --csv output.csv

The tool will:
- Open each file, one by one, in the interactive annotation GUI.
- Let you draw a line to measure the desired length.
- Save the measurement to a results table (CSV or DataFrame).

3. **Calibrate if needed:**  
If you do not specify a pixel-to-millimeter conversion, the tool will prompt you to calibrate using the first file by drawing a line of known real-world length (e.g., 50 mm).

4. **Review and save your results:**  
All measured values (both in pixels and converted to real units if calibrated) are stored in a table, which you can save as a CSV file or use directly in Python for further analysis.

5. **Flexible integration:**  
You can also use the batch measurement tool within your own Python scripts or Jupyter notebooks, receiving results as a pandas DataFrame for seamless integration with your workflow.

This batch measurement functionality makes it fast and reliable to collect manual measurement data, ensuring repeatability and transparency in your annotation workflow.


In [ ]:
# Import the function
from atracker.batch_measure import batch_measure

# Run the batch measurement interactively
sizedata = batch_measure(
    folder="/Users/Jolle/Desktop/test2",
    px_per_mm=20,         # or provide a value, e.g. 5.0
    #conversion_image_mm=50, # known length in mm for calibration (only needed if px_per_mm is None)
    ask_id=False,            # prompt for custom IDs, or set False for filenames
    out_csv="/Users/Jolle/Desktop/measured_lengths.csv",
    ids=["sample1", "sample2", "sample3"]  # <-- optional, must match file count
)

In [ ]:
AT.set_interactive(conv=True, conv_mm=(300), inds=[2])

If you already know your IDs (e.g., a list ["F01", "F02", ..., "F73"]) and want to assign them to your measured files automatically, you can set `ask_id=False` in your batch measurement function. Then after collecting all measurements, simply assign your IDs list as a new column in your DataFrame (or directly as you build the data):

In [ ]:
ids = ["F01","F02"] 
sizedata['ID'] = ids

You can also provide other functions to provide the list, e.g.

In [ ]:
ids = [f"F{str(i).zfill(2)}" for i in range(1, 74)]
print(ids)